# Rzutowanie 4D i teserakt

Samodzielny notebook do pokazania, jak obiekt 4D mozna rzutowac do 3D i 2D. To zachowuje lokalny pomysl dydaktyczny z zajec bez kopiowania zewnetrznych materialow.

In [ ]:
from itertools import combinations, product

import numpy as np
import matplotlib.pyplot as plt

## Wierzcholki i krawedzie hiperszescianu

Teserakt ma 16 wierzcholkow. Dwa wierzcholki laczymy krawedzia, jezeli roznia sie dokladnie jedna wspolrzedna.

In [ ]:
vertices4 = np.array(list(product([-1.0, 1.0], repeat=4)))

edges = []
for i, j in combinations(range(len(vertices4)), 2):
    if np.count_nonzero(vertices4[i] != vertices4[j]) == 1:
        edges.append((i, j))

len(vertices4), len(edges)

## Obrot w 4D

W 4D obrot wybiera plaszczyzne, np. `xw` albo `yz`, a nie pojedyncza os.

In [ ]:
def rotation4(axis_a: int, axis_b: int, angle: float) -> np.ndarray:
    """Macierz obrotu w plaszczyznie wyznaczonej przez dwie osie 4D."""
    R = np.eye(4)
    c, s = np.cos(angle), np.sin(angle)
    R[axis_a, axis_a] = c
    R[axis_b, axis_b] = c
    R[axis_a, axis_b] = -s
    R[axis_b, axis_a] = s
    return R


def rotate4(points: np.ndarray, xy=0.0, xw=0.0, yz=0.0) -> np.ndarray:
    R = rotation4(0, 1, xy) @ rotation4(0, 3, xw) @ rotation4(1, 2, yz)
    return points @ R.T

## Rzut perspektywiczny 4D -> 3D

In [ ]:
def project_4d_to_3d(points: np.ndarray, distance: float = 4.0) -> np.ndarray:
    w = points[:, 3]
    scale = distance / (distance - w)
    return points[:, :3] * scale[:, None]


def plot_tesseract_3d(points3: np.ndarray, title="Teserakt: rzut 4D -> 3D"):
    fig = plt.figure(figsize=(7, 6))
    ax = fig.add_subplot(projection="3d")

    for i, j in edges:
        segment = points3[[i, j]]
        ax.plot(segment[:, 0], segment[:, 1], segment[:, 2], color="black", linewidth=1.4)

    ax.scatter(points3[:, 0], points3[:, 1], points3[:, 2], s=45, color="tab:red")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_zlabel("z")
    ax.set_title(title)
    ax.set_box_aspect((1, 1, 1))
    return ax

In [ ]:
rotated = rotate4(vertices4, xy=0.4, xw=0.9, yz=0.2)
projected3 = project_4d_to_3d(rotated)
plot_tesseract_3d(projected3);

## Rzut 3D -> 2D

In [ ]:
def project_3d_to_2d(points: np.ndarray, distance: float = 5.0) -> np.ndarray:
    z = points[:, 2]
    scale = distance / (distance - z)
    return points[:, :2] * scale[:, None]


def plot_tesseract_2d(points2: np.ndarray, title="Teserakt: rzut 4D -> 2D"):
    fig, ax = plt.subplots(figsize=(6, 6))
    for i, j in edges:
        segment = points2[[i, j]]
        ax.plot(segment[:, 0], segment[:, 1], color="black", linewidth=1.4)

    ax.scatter(points2[:, 0], points2[:, 1], s=45, color="tab:red")
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_title(title)
    return ax

In [ ]:
projected2 = project_3d_to_2d(projected3)
plot_tesseract_2d(projected2);

## Interaktywna wersja w Plotly

In [ ]:
try:
    import plotly.graph_objects as go
except ImportError:
    go = None


def plot_tesseract_plotly(angle=0.8):
    if go is None:
        print("Plotly nie jest zainstalowane w tym srodowisku.")
        return None

    points3 = project_4d_to_3d(rotate4(vertices4, xy=angle, xw=angle / 2, yz=angle / 3))
    x_edges, y_edges, z_edges = [], [], []
    for i, j in edges:
        segment = points3[[i, j]]
        x_edges += [segment[0, 0], segment[1, 0], None]
        y_edges += [segment[0, 1], segment[1, 1], None]
        z_edges += [segment[0, 2], segment[1, 2], None]

    fig = go.Figure()
    fig.add_trace(go.Scatter3d(x=x_edges, y=y_edges, z=z_edges, mode="lines", line=dict(width=4)))
    fig.add_trace(go.Scatter3d(x=points3[:, 0], y=points3[:, 1], z=points3[:, 2], mode="markers"))
    fig.update_layout(width=750, height=650, scene=dict(aspectmode="cube"), title="Teserakt: obrot 4D")
    return fig


plot_tesseract_plotly()

## Mini-zadania

1. Zmien plaszczyzne obrotu w `rotate4` i porownaj rzut.
2. Zmien `distance` w `project_4d_to_3d`; sprawdz efekt perspektywy.
3. Dodaj suwak `ipywidgets.interact` dla parametru `angle`.